# Lesson 01 - Your first MAF agent

**Level:** beginner | **Demo time:** 7 minutes | **Builds on:** Lesson 00

A model client knows how to call a model. An `Agent` adds an identity, instructions, optional tools, middleware, and conversation behavior.

## Learning objectives

- Create an application-owned agent backed by a Foundry model deployment.
- Compare complete and streaming responses.
- Explain which configuration lives in code.

## Application-owned agent

In this pattern, the notebook owns the agent definition:

- **Foundry owns:** the project, model deployment, inference endpoint, and access control.
- **Our code owns:** the agent name, instructions, tools, and session object.

This is the recommended starting point when agent behavior should change with application code.

In [ ]:
from agent_framework import Agent

from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)

settings = WorkshopSettings.from_env()
credential = create_credential()
chat_client = create_chat_client(settings, credential)
print(f"Settings loaded successfully for project at {settings.project_endpoint}")

In [ ]:
support_agent = Agent(
    client=chat_client,
    name="support_concierge",
    description="Front door for Contoso customer support.",
    instructions=(
        "You are a friendly Contoso customer-support concierge. "
        "Give concise, practical answers. Do not invent order status, policies, "
        "or actions that you cannot verify."
    ),
)

print(support_agent.name, "is ready")

The instructions establish role, tone, and boundaries. Notice the explicit rule against inventing business data. In later lessons, tools will give the agent verified data instead.

In [ ]:
question = "My headphones arrived damaged. What information should I gather before contacting support?"
response = await support_agent.run(question)

print("Customer:", question)
print("Agent:", response.text)

`run()` returns an `AgentResponse`. `.text` is the convenient combined text, while `.messages`, `.contents`, `.usage_details`, and `.value` expose richer information for applications.

In [ ]:
streaming_question = "Give me a three-step checklist for describing a damaged delivery."
print("Customer:", streaming_question)
print("Agent: ", end="")

async for update in support_agent.run(streaming_question, stream=True):
    if update.text:
        print(update.text, end="", flush=True)

print()

## Complete vs. streaming

- Use a **complete response** for background jobs and structured post-processing.
- Use **streaming** for conversational user interfaces where perceived latency matters.
- Both calls use the same agent definition; only response delivery changes.

In [ ]:
credential.close()

## Recap

The agent can reason and communicate, but it cannot inspect an order or retrieve a real policy. Lesson 02 adds safe, deterministic Python tools.